# House Prices - EDA
Kaggle "House Prices: Advanced Regression Techniques"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')
train.head()

## 1. 基本統計量と欠損値

In [ ]:
# 欠損値の確認
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()
missing = pd.DataFrame({
    'train_missing': missing_train[missing_train > 0],
    'train_pct': (missing_train[missing_train > 0] / len(train) * 100).round(1),
    'test_missing': missing_test.reindex(missing_train[missing_train > 0].index).fillna(0).astype(int),
    'test_pct': (missing_test.reindex(missing_train[missing_train > 0].index).fillna(0) / len(test) * 100).round(1)
}).sort_values('train_pct', ascending=False)

# testにだけ欠損があるカラムも追加
test_only_missing = missing_test[(missing_test > 0) & (~missing_test.index.isin(missing.index))]
if len(test_only_missing) > 0:
    extra = pd.DataFrame({
        'train_missing': 0,
        'train_pct': 0.0,
        'test_missing': test_only_missing,
        'test_pct': (test_only_missing / len(test) * 100).round(1)
    })
    missing = pd.concat([missing, extra]).sort_values('train_pct', ascending=False)

print(f'欠損があるカラム数: train={sum(missing_train > 0)}, test={sum(missing_test > 0)}')
missing

In [ ]:
# データ型の確認
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print(f'数値カラム: {len(num_cols)}, カテゴリカルカラム: {len(cat_cols)}')
print(f'\n数値カラム: {num_cols}')
print(f'\nカテゴリカルカラム: {cat_cols}')

## 2. 目的変数 SalePrice の分布分析

**なぜ対数変換が必要なのか？**

コンペの評価指標は **RMSLE** (Root Mean Squared Logarithmic Error)。  
これは「絶対額の誤差」ではなく「比率の誤差」を測る指標。

- 1,000万円の家を100万円ズレて予測 → 誤差率10%
- 1億円の家を100万円ズレて予測 → 誤差率1%

MSEで学習すると高額物件の100万円誤差を過剰に罰し、低額物件の100万円誤差を軽視してしまう。  
log変換すると「比率」空間で学習するので、全価格帯で公平に誤差を評価できる。

In [ ]:
price = train['SalePrice']
log_price = np.log1p(price)

# --- 統計量の計算 ---
stats_table = pd.DataFrame({
    'SalePrice (原系列)': [
        price.mean(), price.median(), price.std(),
        price.skew(), price.kurtosis(),
        stats.shapiro(price.sample(500, random_state=42))[1]
    ],
    'log1p(SalePrice)': [
        log_price.mean(), log_price.median(), log_price.std(),
        log_price.skew(), log_price.kurtosis(),
        stats.shapiro(log_price.sample(500, random_state=42))[1]
    ],
    '正規分布 (理論値)': ['-', '-', '-', 0.0, 0.0, '>0.05']
}, index=['Mean', 'Median', 'Std', 'Skewness', 'Kurtosis', 'Shapiro-Wilk p値'])

print('=== SalePrice 分布の統計量 ===')
print(stats_table.to_string())
print()

# 歪度の解釈
skew_orig = price.skew()
skew_log = log_price.skew()
kurt_orig = price.kurtosis()
kurt_log = log_price.kurtosis()

print(f'--- 歪度 (Skewness) ---')
print(f'  原系列: {skew_orig:.4f} → 正の歪み: 右裾が長い (高額物件が少数存在)')
print(f'  log変換後: {skew_log:.4f} → ほぼ対称 (|skew| < 0.5 なら概ね対称)')
print()
print(f'--- 尖度 (Kurtosis) ---')
print(f'  原系列: {kurt_orig:.4f} → 正規分布(0)より尖っている → 外れ値が多い')
print(f'  log変換後: {kurt_log:.4f} → 正規分布に近い')
print()

# Shapiro-Wilk検定
_, p_orig = stats.shapiro(price.sample(500, random_state=42))
_, p_log = stats.shapiro(log_price.sample(500, random_state=42))
print(f'--- Shapiro-Wilk 正規性検定 (n=500) ---')
print(f'  原系列: p={p_orig:.6f} {"→ 正規分布を棄却 (p<0.05)" if p_orig < 0.05 else "→ 正規分布を棄却できない"}')
print(f'  log変換後: p={p_log:.6f} {"→ 正規分布を棄却 (p<0.05)" if p_log < 0.05 else "→ 正規分布を棄却できない"}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# === 上段: 原系列 ===
# ヒストグラム + 正規分布フィット
ax = axes[0, 0]
ax.hist(price, bins=50, density=True, edgecolor='black', alpha=0.7, color='steelblue')
xmin, xmax = price.min(), price.max()
x = np.linspace(xmin, xmax, 200)
mu, sigma = price.mean(), price.std()
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'N({mu:.0f}, {sigma:.0f})')
ax.set_title(f'SalePrice (Skew={skew_orig:.2f}, Kurt={kurt_orig:.2f})')
ax.set_xlabel('SalePrice ($)')
ax.legend()
ax.axvline(price.median(), color='green', ls='--', lw=1.5, label=f'Median={price.median():.0f}')
ax.legend()

# QQ plot
ax = axes[0, 1]
stats.probplot(price, plot=ax)
ax.set_title('QQ Plot (原系列)')
ax.get_lines()[0].set_markersize(3)

# Box plot
ax = axes[0, 2]
bp = ax.boxplot(price, vert=True, widths=0.5, patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
ax.set_title(f'Box Plot (IQR外れ値: {len(price[price > price.quantile(0.75) + 1.5*(price.quantile(0.75)-price.quantile(0.25))])}件)')
ax.set_ylabel('SalePrice ($)')
ax.set_xticklabels(['SalePrice'])

# === 下段: log変換後 ===
# ヒストグラム + 正規分布フィット
ax = axes[1, 0]
ax.hist(log_price, bins=50, density=True, edgecolor='black', alpha=0.7, color='orange')
x = np.linspace(log_price.min(), log_price.max(), 200)
mu_log, sigma_log = log_price.mean(), log_price.std()
ax.plot(x, stats.norm.pdf(x, mu_log, sigma_log), 'r-', lw=2, label=f'N({mu_log:.2f}, {sigma_log:.2f})')
ax.set_title(f'log1p(SalePrice) (Skew={skew_log:.2f}, Kurt={kurt_log:.2f})')
ax.set_xlabel('log1p(SalePrice)')
ax.axvline(log_price.median(), color='green', ls='--', lw=1.5)
ax.legend()

# QQ plot
ax = axes[1, 1]
stats.probplot(log_price, plot=ax)
ax.set_title('QQ Plot (log変換後)')
ax.get_lines()[0].set_markersize(3)

# Box plot
ax = axes[1, 2]
bp = ax.boxplot(log_price, vert=True, widths=0.5, patch_artist=True)
bp['boxes'][0].set_facecolor('orange')
bp['boxes'][0].set_alpha(0.7)
ax.set_title('Box Plot (log変換後)')
ax.set_ylabel('log1p(SalePrice)')
ax.set_xticklabels(['log1p(SalePrice)'])

plt.tight_layout()
plt.savefig('figures/saleprice_distribution_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 対数変換の効果: 誤差の「公平性」を可視化 ===
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左: MSE空間 (原系列) — 同じ$20,000の誤差の影響
ax = axes[0]
prices_example = np.array([100000, 200000, 400000, 600000])
error = 20000  # 全物件で同じ$20,000の誤差
error_pct = error / prices_example * 100

bars = ax.bar(range(len(prices_example)),
              [error]*len(prices_example),
              tick_label=[f'{p//1000}k' for p in prices_example],
              color='steelblue', alpha=0.7, edgecolor='black')
ax2 = ax.twinx()
ax2.plot(range(len(prices_example)), error_pct, 'ro-', ms=8, lw=2)
ax2.set_ylabel('Error Rate (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax.set_ylabel('Absolute Error')
ax.set_xlabel('House Price')
ax.set_title('MSE: Same 20k error penalized equally\n100k house=20%, 600k house=3.3%')

# 右: RMSLE空間 (log変換後) — 同じ10%の誤差の影響
ax = axes[1]
log_error_10pct = np.log1p(prices_example * 1.1) - np.log1p(prices_example)
ax.bar(range(len(prices_example)),
       log_error_10pct,
       tick_label=[f'{p//1000}k' for p in prices_example],
       color='orange', alpha=0.7, edgecolor='black')
ax.set_ylabel('Error in log space')
ax.set_xlabel('House Price')
ax.set_title('RMSLE: 10% error penalized equally\nFair evaluation across all price ranges')

plt.tight_layout()
plt.savefig('figures/log_transform_fairness.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== 対数変換が必要な3つの理由 ===')
print()
print('1. 評価指標との整合性:')
print('   RMSLE = RMSE(log(pred), log(actual))')
print('   → log1p(SalePrice) を目的変数にすれば、MSE最小化 = RMSLE最小化')
print()
print('2. 誤差の公平性:')
print(f'   100k の家を 20k ズレて予測: 誤差率 {20/100*100:.0f}%')
print(f'   600k の家を 20k ズレて予測: 誤差率 {20/600*100:.1f}%')
print('   → MSEではどちらも同じペナルティ。RMSLEは比率で評価するので公平')
print()
print('3. 分布の正規化:')
print(f'   原系列: Skew={skew_orig:.2f}, Kurt={kurt_orig:.2f} (右に大きく歪む)')
print(f'   log後:  Skew={skew_log:.2f}, Kurt={kurt_log:.2f} (ほぼ正規分布)')
print('   → Ridge/Lasso等の線形モデルは正規分布の仮定で性能が出る')

## 3. 数値特徴量 vs SalePrice

In [ ]:
# SalePriceとの相関
corr = train[num_cols].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)
print('Top 15 positive correlations with SalePrice:')
print(corr.head(15))
print(f'\nTop 5 negative correlations:')
print(corr.tail(5))

In [ ]:
# 上位10特徴量の散布図
top_features = corr.abs().sort_values(ascending=False).head(10).index.tolist()
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(top_features):
    ax = axes[i // 5, i % 5]
    ax.scatter(train[feat], train['SalePrice'], alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')
    r = train[feat].corr(train['SalePrice'])
    ax.set_title(f'r={r:.3f}')
plt.tight_layout()
plt.savefig('figures/top10_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. カテゴリカル特徴量 vs SalePrice

In [ ]:
# カテゴリカル特徴量ごとのSalePrice中央値
cat_importance = {}
for col in cat_cols:
    groups = train.groupby(col)['SalePrice'].median()
    cat_importance[col] = groups.std()  # グループ間分散で重要度を推定

cat_imp = pd.Series(cat_importance).sort_values(ascending=False)
print('カテゴリカル特徴量の重要度 (グループ間SalePrice中央値のstd):')
print(cat_imp.head(15))

In [ ]:
# 上位6カテゴリカル特徴量のboxplot
top_cats = cat_imp.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(top_cats):
    ax = axes[i // 3, i % 3]
    order = train.groupby(col)['SalePrice'].median().sort_values().index
    sns.boxplot(data=train, x=col, y='SalePrice', order=order, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('figures/top6_categorical_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.1 Neighborhood 別 SalePrice 分析

Neighborhoodは25カテゴリ。Label Encodingではアルファベット順に0-24が割り振られ、
「価格帯の情報」が失われる。Target Encodingでカテゴリを「そのカテゴリの期待SalePrice」に
置き換えれば、線形モデルでも地域の価格差を直接学習できる。

**リスク**: train全体の平均をそのまま使うとリーク。fold内でLOO (Leave-One-Out) で計算する。

In [ ]:
# === Neighborhood 別 log1p(SalePrice) の箱ひげ図 ===
train['LogSalePrice'] = np.log1p(train['SalePrice'])

nb_stats = train.groupby('Neighborhood')['LogSalePrice'].agg(['mean', 'median', 'std', 'count'])
nb_stats = nb_stats.sort_values('median')

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

# 上: 箱ひげ図
ax = axes[0]
order = nb_stats.index.tolist()
bp = sns.boxplot(data=train, x='Neighborhood', y='LogSalePrice', order=order, ax=ax,
                 palette='coolwarm')
ax.set_title('Neighborhood vs log1p(SalePrice)')
ax.set_ylabel('log1p(SalePrice)')
ax.tick_params(axis='x', rotation=75)
# 全体平均の水平線
ax.axhline(train['LogSalePrice'].mean(), color='red', ls='--', lw=1.5, alpha=0.7, label='Global Mean')
ax.legend()

# 下: サンプル数
ax = axes[1]
ax.bar(range(len(order)), nb_stats.loc[order, 'count'], color='steelblue', alpha=0.7)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=75, fontsize=8)
ax.set_ylabel('Count')
ax.set_title('Sample size per Neighborhood')

plt.tight_layout()
plt.savefig('figures/neighborhood_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

# 統計量の表示
print('=== Neighborhood 別 log1p(SalePrice) 統計量 ===')
print(nb_stats.round(3).to_string())
print(f'\n全体平均: {train["LogSalePrice"].mean():.3f}')
print(f'Neighborhood間の中央値の範囲: {nb_stats["median"].min():.3f} ~ {nb_stats["median"].max():.3f}')
print(f'  → 差分: {nb_stats["median"].max() - nb_stats["median"].min():.3f}')
print(f'  → 元スケールで {np.expm1(nb_stats["median"].min()):.0f} ~ {np.expm1(nb_stats["median"].max()):.0f}')

# ANOVA検定
from scipy.stats import f_oneway
groups = [g['LogSalePrice'].values for _, g in train.groupby('Neighborhood')]
f_stat, p_val = f_oneway(*groups)
print(f'\n一元配置分散分析 (ANOVA): F={f_stat:.1f}, p={p_val:.2e}')
print(f'  → {"Neighborhood間に統計的に有意な差あり (p<0.05)" if p_val < 0.05 else "有意差なし"}')

# Label Encoding vs Target Encoding の違い
print(f'\n=== Label Encoding の問題 ===')
print('Label Encodingではアルファベット順: Blmngtn=0, ..., Veenker=24')
print('→ 「NoRidgeは高い、MeadowVは安い」という情報が数値に反映されない')
print('→ tree系モデルは分割で対応可能だが、線形モデルでは致命的')

train.drop(columns=['LogSalePrice'], inplace=True)

## 5. 特徴量間の相関 (多重共線性チェック)

In [ ]:
# 高相関ペアの検出
corr_matrix = train[num_cols].corr()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature1', 'Feature2', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', ascending=False)
print(f'相関 > 0.7 のペア数: {len(high_corr_df)}')
high_corr_df

## 6. 外れ値の検出

In [ ]:
# GrLivArea vs SalePrice (外れ値の定番)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(train['GrLivArea'], train['SalePrice'], alpha=0.5, s=15)
ax.set_xlabel('GrLivArea')
ax.set_ylabel('SalePrice')
ax.set_title('GrLivArea vs SalePrice (全データ)')

# 外れ値候補をハイライト
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'], c='red', s=50, label=f'Outliers ({len(outliers)})')
ax.legend()

# 外れ値除去後
train_clean = train.drop(outliers.index)
ax = axes[1]
ax.scatter(train_clean['GrLivArea'], train_clean['SalePrice'], alpha=0.5, s=15)
ax.set_xlabel('GrLivArea')
ax.set_ylabel('SalePrice')
ax.set_title('GrLivArea vs SalePrice (外れ値除去後)')

plt.tight_layout()
plt.savefig('figures/outliers_grlivarea.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'外れ値候補: {outliers[["Id", "GrLivArea", "SalePrice"]].to_string()}')

## 7. まとめ

In [ ]:
print('=== EDA まとめ ===')
print(f'Train: {train.shape[0]}行 × {train.shape[1]}列')
print(f'Test: {test.shape[0]}行 × {test.shape[1]}列')
print(f'数値特徴量: {len(num_cols)}, カテゴリカル特徴量: {len(cat_cols)}')
print(f'\nSalePrice: mean={train["SalePrice"].mean():.0f}, median={train["SalePrice"].median():.0f}')
print(f'SalePrice skew: {skew_orig:.2f} → log1p後: {skew_log:.2f}')
print(f'\n高相関ペア (>0.7): {len(high_corr_df)}組')
print(f'外れ値候補 (GrLivArea>4000 & 低価格): {len(outliers)}件')
print(f'\n=== 方針 ===')
print('1. SalePriceはlog1p変換して学習 (歪みが強い → RMSLE評価に最適)')
print('2. GrLivAreaの外れ値2件は除去')
print('3. NA = "なし" のカテゴリカル特徴量は適切にfill')
print('4. 順序カテゴリ(Ex/Gd/TA/Fa/Po等)は数値エンコード')